# Multi-model benchmark results

Load Oolong Real and BrowseComp-Plus `multi_model_benchmarks/**/results*.jsonl` into one pandas `DataFrame`.

- One row per trial (no aggregation).
- Skips `latest.jsonl` (symlink).
- **Write-back:** after editing the table (e.g. `success`), call `write_multi_model_benchmark_results(df)` to overwrite each `source_file` using **only** the canonical JSONL columns (no `benchmark` / `source_file` on disk). If you edit JSONL by hand, use **JSON** syntax: `true` / `false` / `null` (not Python `True` / `False` / `None`).
- Requires **pandas**: `uv pip install pandas`

**Run this notebook with the working directory set to the `rlm` repository root** (or set `REPO_ROOT` in the next cell).

In [3]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "bench_Oolong_real").is_dir():
    raise RuntimeError(
        "Set the notebook working directory to the rlm repo root, or assign REPO_ROOT manually."
    )

In [5]:
import json


def _jsonl_paths_for_benchmark(
    multi_root: Path,
    *,
    include_stable: bool,
    include_timestamped: bool,
) -> list[Path]:
    if not multi_root.is_dir():
        return []
    out: list[Path] = []
    for baseline_dir in sorted(p for p in multi_root.iterdir() if p.is_dir()):
        if include_stable:
            stable = baseline_dir / "results.jsonl"
            if stable.is_file():
                out.append(stable)
        if include_timestamped:
            out.extend(sorted(baseline_dir.glob("results_*.jsonl")))
    return out


def discover_result_paths(
    *,
    repo_root: Path | None = None,
    include_stable: bool = True,
    include_timestamped: bool = False,
) -> list[tuple[str, Path]]:
    root = repo_root or REPO_ROOT
    oolong = root / "bench_Oolong_real" / "multi_model_benchmarks"
    browse = root / "bench_BrowseComp-Plus" / "multi_model_benchmarks"
    pairs: list[tuple[str, Path]] = []
    for label, multi in (
        ("oolong_real", oolong),
        ("browsecomp_plus", browse),
    ):
        for p in _jsonl_paths_for_benchmark(
            multi,
            include_stable=include_stable,
            include_timestamped=include_timestamped,
        ):
            pairs.append((label, p))
    return pairs


def _load_one_jsonl(benchmark: str, path: Path) -> pd.DataFrame:
    rows: list[dict] = []
    text = path.read_text(encoding="utf-8")
    for lineno, line in enumerate(text.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Invalid JSON in {path} line {lineno}: {e.msg}\n"
                f"Use JSON booleans true/false and null (not Python True/False/None)."
            ) from e
    df = pd.DataFrame(rows)
    df.insert(0, "benchmark", benchmark)
    df["source_file"] = str(path)
    return df


def load_multi_model_benchmark_results(
    *,
    repo_root: Path | None = None,
    include_stable: bool = True,
    include_timestamped: bool = False,
) -> pd.DataFrame:
    pairs = discover_result_paths(
        repo_root=repo_root,
        include_stable=include_stable,
        include_timestamped=include_timestamped,
    )
    if not pairs:
        return pd.DataFrame()
    frames = [_load_one_jsonl(b, p) for b, p in pairs]
    return pd.concat(frames, ignore_index=True)

In [14]:
df = load_multi_model_benchmark_results()

In [11]:


print(f"rows={len(df)}  columns={list(df.columns)}")
#display(df)

rows=60  columns=['benchmark', 'task_id', 'query', 'baseline', 'trial', 'ground_truth', 'response', 'success', 'subagent_calls', 'total_time', 'input_tokens', 'source_file']


In [16]:
def _get_agg_spec(dataframe):
    """
    Returns a valid pandas aggregation spec mapping.
    Avoids passing None or empty keys to .agg()
    """
    spec = {
        "num_trials": ("trial", "nunique") if "trial" in dataframe.columns else ("task_id", "count"),
        "num_tasks": ("task_id", "nunique"),
    }
    if "score" in dataframe.columns:
        spec["mean_score"] = ("score", "mean")
    if "success" in dataframe.columns:
        spec["success_rate"] = ("success", "mean")
    if "total_time" in dataframe.columns:
        spec["avg_time_sec"] = ("total_time", "mean")
    if "input_tokens" in dataframe.columns:
        spec["avg_input_tokens"] = ("input_tokens", "mean")
        
    return spec

# 1. Determine which grouping columns are available
group_cols = []
if "benchmark" in df.columns:
    group_cols.append("benchmark")
if "baseline" in df.columns:
    group_cols.append("baseline")

# 2. Perform the combined aggregation
if group_cols:
    # We use a list [group_cols] to group by multiple levels simultaneously
    combined_agg = df.groupby(group_cols).agg(**_get_agg_spec(df)).round(2)
    
    print(f"Aggregate by {(' and ').join(group_cols)}:")
    display(combined_agg)
    
    # Optional: If you want to see the table flattened (no multi-index)
    # display(combined_agg.reset_index())
else:
    print("Neither 'benchmark' nor 'baseline' columns found in the DataFrame.")

Aggregate by benchmark and baseline:


num_trials  num_tasks  success_rate  \
benchmark       baseline                                                      
browsecomp_plus flagship-root_nano-sub           2          5           0.8   
                mini-root_mini-sub               2          5           0.5   
                mini-root_nano-sub               2          5           0.4   
oolong_real     flagship-root_nano-sub           2          5           0.4   
                mini-root_mini-sub               2          5           0.0   
                mini-root_nano-sub               2          5           0.1   

                                        avg_time_sec  avg_input_tokens  
benchmark       baseline                                                
browsecomp_plus flagship-root_nano-sub         63.16          185205.7  
                mini-root_mini-sub             39.91          262435.2  
                mini-root_nano-sub             36.51          228589.5  
oolong_real     flagship-root_nano-sub         35.41           42020.7  
                mini-root_mini-sub             12.57           38079.9  
                mini-root_nano-sub             10.40           27395.9